In [ ]:
# # 05b. Monte Carlo uncertainty post-processing, local run
# 
# This notebook reads the server outputs from 05a and produces submission-ready uncertainty tables.
# 

# -*- coding: utf-8 -*-
from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def find_submission_dir(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == "submission_code":
            return candidate
        if (candidate / "submission_code").is_dir():
            return candidate / "submission_code"
    if start.name == "5_uncertainty_analysis":
        return start.parent
    return start


SUBMISSION_DIR = find_submission_dir()
NOTEBOOK_DIR = SUBMISSION_DIR / "5_uncertainty_analysis"
INPUT_DIR = SUBMISSION_DIR / "input"
SERVER_OUTPUT = NOTEBOOK_DIR / "server_output"
OUTPUT_DIR = SUBMISSION_DIR / "output" / "uncertainty_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HOUSEHOLD_DIR = INPUT_DIR / "household_expenditure"
REFERENCE_DIR = INPUT_DIR / "reference"
SCENARIO_DIR = SUBMISSION_DIR / "output" / "scenario_analysis"
THEIL_DIR = SUBMISSION_DIR / "3_ThileT_and_shapely_decomposation" / "output"

FILES = {
    "fp_by_gloria": SERVER_OUTPUT / "fp_by_gloria_baseline.npy",
    "sat_by_lu": SERVER_OUTPUT / "sat_by_landuse_2023.npy",
    "sigma": SERVER_OUTPUT / "cf_sigma_landuse.npy",
    "mc": SERVER_OUTPUT / "mc_factors_independent.npy",
    "category_results": SERVER_OUTPUT / "uncertainty_category_results.xlsx",
    "population": HOUSEHOLD_DIR / "Population_by_IncomeGroup.csv",
    "country_mapping": HOUSEHOLD_DIR / "GLORIA_Country_Mapping.csv",
    "country_class": REFERENCE_DIR / "nationlist_categorized_titlecase.csv",
    "scenario_long": SCENARIO_DIR / "scenario_footprint_long_2023.csv",
    "theil_shapley": THEIL_DIR / "theil_shapley_summary_2023.xlsx",
}

G, S, B = 164, 120, 201
GS = G * S
r_index = np.arange(GS) // S
CHUNK_BASELINE = 2000
CHUNK_MC = 200

t0 = time.time()


def log(message):
    print(f"[{time.time() - t0:7.0f} s] {message}", flush=True)


def require_files(paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError("Missing required files. Run 05a on the server first and copy its server_output folder here.\n  " + "\n  ".join(missing))


def ci(values):
    values = np.asarray(values, float)
    return float(np.median(values)), float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))


def compute_gini(fp, pop):
    fp = np.asarray(fp, float).ravel()
    pop = np.asarray(pop, float).ravel()
    mask = pop > 0
    fp, pop = fp[mask], pop[mask]
    idx = np.argsort(fp / pop)
    fp, pop = fp[idx], pop[idx]
    lx = np.concatenate([[0.0], np.cumsum(pop)]) / pop.sum()
    ly = np.concatenate([[0.0], np.cumsum(fp)]) / fp.sum()
    return float(1 - 2 * np.trapz(ly, lx))


def country_gini(fp_b, pop_b):
    mask = pop_b > 0
    if mask.sum() < 2:
        return np.nan
    return compute_gini(fp_b[mask], pop_b[mask])


def compute_theil(fp_jb, pop_jb):
    fp_jb = np.asarray(fp_jb, float)
    pop_jb = np.asarray(pop_jb, float)
    pop_j = pop_jb.sum(axis=1)
    fp_j = fp_jb.sum(axis=1)
    mu = fp_j.sum() / pop_j.sum()
    pc_j = np.where(pop_j > 0, fp_j / pop_j, 0.0)
    r_j = np.where(mu > 0, pc_j / mu, 0.0)
    w_j = pop_j / pop_j.sum()
    between = float(np.sum(w_j * r_j * np.where(r_j > 0, np.log(r_j), 0.0)))
    pc_jb = np.where(pop_jb > 0, fp_jb / pop_jb, 0.0)
    r_jb = np.where(pc_j[:, None] > 0, pc_jb / pc_j[:, None], 0.0)
    w_jb = np.where(pop_j[:, None] > 0, pop_jb / pop_j[:, None], 0.0)
    within_j = (w_jb * r_jb * np.where(r_jb > 0, np.log(r_jb), 0.0)).sum(axis=1)
    within = float(np.sum(w_j * r_j * within_j))
    return between + within, between, within


def theil_flat(fp_1d, pop_1d):
    fp_1d = np.asarray(fp_1d, float)
    pop_1d = np.asarray(pop_1d, float)
    mask = pop_1d > 0
    if mask.sum() < 2:
        return np.nan
    fp, pop = fp_1d[mask], pop_1d[mask]
    mu = fp.sum() / pop.sum()
    pc = fp / pop
    ratio = np.where(mu > 0, pc / mu, 0.0)
    weight = pop / pop.sum()
    return float(np.sum(weight * ratio * np.where(ratio > 0, np.log(ratio), 0.0)))


require_files([
    FILES["fp_by_gloria"],
    FILES["sat_by_lu"],
    FILES["sigma"],
    FILES["mc"],
    FILES["category_results"],
    FILES["population"],
    FILES["country_mapping"],
    FILES["country_class"],
])

print("Submission directory:", SUBMISSION_DIR)
print("Outputs:", OUTPUT_DIR)

# Part 1. Load baseline tensor, population, classifications, and MC inputs.
log("Streaming fp_by_gloria_baseline with memory mapping")
fpg_mm = np.load(FILES["fp_by_gloria"], mmap_mode="r")
assert fpg_mm.shape == (GS, G, B), f"Unexpected fp_by_gloria shape: {fpg_mm.shape}"
fp_flat = np.empty((GS, G * B), dtype=np.float32)
fp_baseline = np.zeros((G, B), dtype=np.float64)
for start in range(0, GS, CHUNK_BASELINE):
    end = min(start + CHUNK_BASELINE, GS)
    chunk = np.asarray(fpg_mm[start:end])
    fp_flat[start:end] = chunk.reshape(end - start, G * B).astype(np.float32)
    fp_baseline += chunk.sum(axis=0)
    del chunk
del fpg_mm
gc.collect()
log(f"Household baseline total PDF = {fp_baseline.sum():.6e}")

sat_by_lu = np.load(FILES["sat_by_lu"]).astype(np.float64)
sat_base = sat_by_lu.sum(axis=0)
sigma_matrix = np.load(FILES["sigma"]).astype(np.float64)
unbias = np.exp(-0.5 * sigma_matrix ** 2)
mc_factors = (np.load(FILES["mc"]) * unbias[None]).astype(np.float32)
N_ITER = mc_factors.shape[0]
print("MC draws:", N_ITER)

pop_raw = pd.read_csv(FILES["population"], index_col=0)
mapping = pd.read_csv(FILES["country_mapping"])
pop = np.zeros((G, B), dtype=np.float64)
for gi in range(G):
    rows = mapping[mapping["GLORIA_Index"] == gi + 1]
    for _, row in rows.iterrows():
        iso3 = row["Population_ISO3"]
        if pd.notna(iso3) and iso3 != "Not in Population" and iso3 in pop_raw.columns:
            pop[gi] += pop_raw[iso3].values
world_pop = pop.sum()

gloria_iso3 = []
for gi in range(G):
    rows = mapping[mapping["GLORIA_Index"] == gi + 1]
    gloria_iso3.append(rows.iloc[0]["ISO3"] if len(rows) else None)

country_class = pd.read_csv(FILES["country_class"]).sort_values("Country_ID").reset_index(drop=True)
country_names = country_class["Country_Name"].values[:G]
wbr = country_class["WBR"].values[:G]
WBR_NAMES = {
    "EAP": "East Asia & Pacific",
    "ECA": "Europe & Central Asia",
    "LAC": "Latin Am. & Caribbean",
    "MENA": "Middle East & N. Africa",
    "NAM": "North America",
    "SAR": "South Asia",
    "SSA": "Sub-Saharan Africa",
}
WBR_CODES = [code for code in sorted(set(wbr)) if pd.notna(code)]
WBR_MASKS = [(wbr == code) for code in WBR_CODES]
print(f"Population total: {world_pop / 1e9:.3f} billion")
print(f"WBR groups: {WBR_CODES}")

# Part 2. Streaming independent Monte Carlo statistics.
def ratios_chunk(mc_chunk):
    out = np.empty((mc_chunk.shape[0], GS), dtype=np.float32)
    for i in range(mc_chunk.shape[0]):
        f_sector = mc_chunk[i][r_index]
        sat_k = np.einsum("ls,sl->s", sat_by_lu, f_sector)
        out[i] = np.where(sat_base > 1e-30, sat_k / sat_base, 1.0).astype(np.float32)
    return out


def load_scenario_alpha():
    if not FILES["scenario_long"].exists():
        print("Scenario file not found; scenario uncertainty will be skipped:", FILES["scenario_long"])
        return {}
    scenario_long = pd.read_csv(FILES["scenario_long"])
    mapping_names = {
        "S1": "S_T10_e",
        "S2": "S_T20_e",
        "S3": "S_T30_e",
        "S4": "S_W10_e",
        "S5": "S_W20_e",
        "S6": "S_W30_e",
    }
    alpha = {}
    base_pc = scenario_long["BF_percap_BASE"].values
    country = scenario_long["Country_ID"].values.astype(int) - 1
    bins = scenario_long["BinIndex"].values.astype(int)
    for scenario, suffix in mapping_names.items():
        col = f"BF_percap_{suffix}"
        if col not in scenario_long.columns:
            continue
        a = np.ones((G, B), dtype=np.float64)
        ratio = np.where(base_pc > 1e-30, scenario_long[col].values / base_pc, 1.0)
        a[country, bins] = ratio
        alpha[scenario] = a
    return alpha


scenario_alpha = load_scenario_alpha()
scenario_names = list(scenario_alpha.keys())
n_wbr = len(WBR_CODES)
sat_base64 = sat_base.astype(np.float64)

total_household = np.empty(N_ITER, dtype=np.float64)
total_all_final_demand = np.empty(N_ITER, dtype=np.float64)
country_total = np.empty((N_ITER, G), dtype=np.float64)
gini_global = np.empty(N_ITER, dtype=np.float64)
theil_global = np.empty((N_ITER, 3), dtype=np.float64)
gini_wbr = np.empty((N_ITER, n_wbr), dtype=np.float64)
theil_wbr = np.empty((N_ITER, n_wbr, 3), dtype=np.float64)
gini_country = np.empty((N_ITER, G), dtype=np.float64)
theil_country = np.empty((N_ITER, G), dtype=np.float64)
scenario_total = {name: np.empty(N_ITER, dtype=np.float64) for name in scenario_names}
scenario_gini = {name: np.empty(N_ITER, dtype=np.float64) for name in scenario_names}

for start in range(0, N_ITER, CHUNK_MC):
    end = min(start + CHUNK_MC, N_ITER)
    ar = ratios_chunk(mc_factors[start:end])
    grid = (ar @ fp_flat).reshape(end - start, G, B).astype(np.float64)
    total_all_final_demand[start:end] = ar.astype(np.float64) @ sat_base64
    total_household[start:end] = grid.sum(axis=(1, 2))
    country_total[start:end] = grid.sum(axis=2)

    for i in range(end - start):
        k = start + i
        draw = grid[i]
        gini_global[k] = compute_gini(draw, pop)
        theil_global[k] = compute_theil(draw, pop)
        for j in range(G):
            gini_country[k, j] = country_gini(draw[j], pop[j])
            theil_country[k, j] = theil_flat(draw[j], pop[j])
        for ri, mask in enumerate(WBR_MASKS):
            gini_wbr[k, ri] = compute_gini(draw[mask], pop[mask])
            theil_wbr[k, ri] = compute_theil(draw[mask], pop[mask])

    for name, alpha in scenario_alpha.items():
        sc_grid = grid * alpha[None]
        scenario_total[name][start:end] = sc_grid.sum(axis=(1, 2))
        for i in range(end - start):
            scenario_gini[name][start + i] = compute_gini(sc_grid[i], pop)
        del sc_grid

    del grid, ar
    gc.collect()
    log(f"Processed draws {end}/{N_ITER}")

del mc_factors
gc.collect()
print("Finished independent MC scan.")

# Part 3. Build footprint, inequality, scenario, sector-category, and optional Shapley-Q tables.
nonhousehold = total_all_final_demand - total_household

global_footprint_rows = []
for component, base, draws in [
    ("Household", fp_baseline.sum(), total_household),
    ("All_final_demand", float(sat_base.sum()), total_all_final_demand),
    ("Non_household", float(sat_base.sum()) - fp_baseline.sum(), nonhousehold),
]:
    med, lo, hi = ci(draws)
    pc_med, pc_lo, pc_hi = ci(draws / world_pop * 1e12)
    global_footprint_rows.append({
        "component": component,
        "FP_total_baseline": base,
        "FP_total_median": med,
        "FP_total_lo95": lo,
        "FP_total_hi95": hi,
        "FP_percap_1e12_baseline": base / world_pop * 1e12,
        "FP_percap_1e12_median": pc_med,
        "FP_percap_1e12_lo95": pc_lo,
        "FP_percap_1e12_hi95": pc_hi,
    })
df_footprint_global = pd.DataFrame(global_footprint_rows)

footprint_wbr_rows = []
for ri, code in enumerate(WBR_CODES):
    mask = WBR_MASKS[ri]
    reg_pop = pop[mask].sum()
    base = fp_baseline[mask].sum()
    draws = country_total[:, mask].sum(axis=1)
    med, lo, hi = ci(draws)
    pc_med, pc_lo, pc_hi = ci(draws / reg_pop * 1e12)
    footprint_wbr_rows.append({
        "WBR": code,
        "Region": WBR_NAMES.get(code, code),
        "FP_total_baseline": base,
        "FP_total_median": med,
        "FP_total_lo95": lo,
        "FP_total_hi95": hi,
        "FP_percap_1e12_baseline": base / reg_pop * 1e12,
        "FP_percap_1e12_median": pc_med,
        "FP_percap_1e12_lo95": pc_lo,
        "FP_percap_1e12_hi95": pc_hi,
    })
df_footprint_wbr = pd.DataFrame(footprint_wbr_rows)

base_country_total = fp_baseline.sum(axis=1)
base_country_pop = pop.sum(axis=1)
cty_med = np.median(country_total, axis=0)
cty_lo, cty_hi = np.percentile(country_total, [2.5, 97.5], axis=0)
country_pc = country_total / base_country_pop[None] * 1e12
pc_med = np.median(country_pc, axis=0)
pc_lo, pc_hi = np.percentile(country_pc, [2.5, 97.5], axis=0)
df_footprint_country = pd.DataFrame({
    "GLORIA_Index": np.arange(1, G + 1),
    "ISO3": gloria_iso3,
    "Country": country_names,
    "WBR": wbr,
    "FP_total_baseline": base_country_total,
    "FP_total_median": cty_med,
    "FP_total_lo95": cty_lo,
    "FP_total_hi95": cty_hi,
    "FP_percap_1e12_baseline": np.where(base_country_pop > 0, base_country_total / base_country_pop * 1e12, np.nan),
    "FP_percap_1e12_median": pc_med,
    "FP_percap_1e12_lo95": pc_lo,
    "FP_percap_1e12_hi95": pc_hi,
})

g_base = compute_gini(fp_baseline, pop)
t_base, tb_base, tw_base = compute_theil(fp_baseline, pop)
g_med, g_lo, g_hi = ci(gini_global)
t_med, t_lo, t_hi = ci(theil_global[:, 0])
tb_med, tb_lo, tb_hi = ci(theil_global[:, 1])
tw_med, tw_lo, tw_hi = ci(theil_global[:, 2])
tb_pct = theil_global[:, 1] / theil_global[:, 0] * 100
tbp_med, tbp_lo, tbp_hi = ci(tb_pct)
df_inequality_global = pd.DataFrame([{
    "scope": "Global",
    "Gini_baseline": g_base,
    "Gini_median": g_med,
    "Gini_lo95": g_lo,
    "Gini_hi95": g_hi,
    "Theil_total_baseline": t_base,
    "Theil_total_median": t_med,
    "Theil_total_lo95": t_lo,
    "Theil_total_hi95": t_hi,
    "Theil_between_baseline": tb_base,
    "Theil_between_median": tb_med,
    "Theil_between_lo95": tb_lo,
    "Theil_between_hi95": tb_hi,
    "Theil_within_baseline": tw_base,
    "Theil_within_median": tw_med,
    "Theil_within_lo95": tw_lo,
    "Theil_within_hi95": tw_hi,
    "Theil_between_pct_baseline": tb_base / t_base * 100,
    "Theil_between_pct_median": tbp_med,
    "Theil_between_pct_lo95": tbp_lo,
    "Theil_between_pct_hi95": tbp_hi,
    "P_between_gt_within": float((theil_global[:, 1] > theil_global[:, 2]).mean() * 100),
}])

inequality_wbr_rows = []
for ri, code in enumerate(WBR_CODES):
    mask = WBR_MASKS[ri]
    gb = compute_gini(fp_baseline[mask], pop[mask])
    tt, tb, tw = compute_theil(fp_baseline[mask], pop[mask])
    gm, glo, ghi = ci(gini_wbr[:, ri])
    tm, tlo, thi = ci(theil_wbr[:, ri, 0])
    tbm, tblo, tbhi = ci(theil_wbr[:, ri, 1])
    twm, twlo, twhi = ci(theil_wbr[:, ri, 2])
    inequality_wbr_rows.append({
        "WBR": code,
        "Region": WBR_NAMES.get(code, code),
        "Gini_baseline": gb,
        "Gini_median": gm,
        "Gini_lo95": glo,
        "Gini_hi95": ghi,
        "Theil_total_baseline": tt,
        "Theil_total_median": tm,
        "Theil_total_lo95": tlo,
        "Theil_total_hi95": thi,
        "Theil_between_baseline": tb,
        "Theil_between_median": tbm,
        "Theil_between_lo95": tblo,
        "Theil_between_hi95": tbhi,
        "Theil_within_baseline": tw,
        "Theil_within_median": twm,
        "Theil_within_lo95": twlo,
        "Theil_within_hi95": twhi,
    })
df_inequality_wbr = pd.DataFrame(inequality_wbr_rows)

gini_country_base = np.array([country_gini(fp_baseline[j], pop[j]) for j in range(G)])
theil_country_base = np.array([theil_flat(fp_baseline[j], pop[j]) for j in range(G)])
gcty_med = np.nanmedian(gini_country, axis=0)
gcty_lo, gcty_hi = np.nanpercentile(gini_country, [2.5, 97.5], axis=0)
tcty_med = np.nanmedian(theil_country, axis=0)
tcty_lo, tcty_hi = np.nanpercentile(theil_country, [2.5, 97.5], axis=0)
df_inequality_country = pd.DataFrame({
    "GLORIA_Index": np.arange(1, G + 1),
    "ISO3": gloria_iso3,
    "Country": country_names,
    "WBR": wbr,
    "Gini_baseline": gini_country_base,
    "Gini_median": gcty_med,
    "Gini_lo95": gcty_lo,
    "Gini_hi95": gcty_hi,
    "Theil_baseline": theil_country_base,
    "Theil_median": tcty_med,
    "Theil_lo95": tcty_lo,
    "Theil_hi95": tcty_hi,
})

scenario_rows = []
for name in scenario_names:
    d_total = (scenario_total[name] / total_household - 1.0) * 100
    d_gini = (scenario_gini[name] / gini_global - 1.0) * 100
    tm, tlo, thi = ci(d_total)
    gm, glo, ghi = ci(d_gini)
    scenario_rows.append({
        "scenario": name,
        "dBF_pct_median": round(tm, 2),
        "dBF_lo95": round(tlo, 2),
        "dBF_hi95": round(thi, 2),
        "dGini_pct_median": round(gm, 2),
        "dGini_lo95": round(glo, 2),
        "dGini_hi95": round(ghi, 2),
        "P_dBF_neg": round(float((d_total < 0).mean() * 100), 1),
        "P_dGini_neg": round(float((d_gini < 0).mean() * 100), 1),
    })
df_scenario = pd.DataFrame(scenario_rows)

df_sector_class = pd.read_excel(FILES["category_results"], sheet_name="Sector_class_Gini_FP")
df_sector_group = pd.read_excel(FILES["category_results"], sheet_name="Sector_5class_Gini_FP")

def optional_shapley_q_tables():
    if not FILES["theil_shapley"].exists():
        print("Theil-Shapley output not found; Shapley-Q uncertainty table will be skipped:", FILES["theil_shapley"])
        return {}
    m1_global = pd.read_excel(FILES["theil_shapley"], sheet_name="Method1_Global")
    m1_country = pd.read_excel(FILES["theil_shapley"], sheet_name="Method1_Country")
    m1_wbr = pd.read_excel(FILES["theil_shapley"], sheet_name="Method1_WBR")

    def comp(df, scope_id=None):
        rows = df.copy()
        if scope_id is not None:
            rows = rows[rows["scope_id"].astype(str) == str(scope_id)]
        return dict(zip(rows["component"], rows["value"].astype(float)))

    global_dict = comp(m1_global)
    stable = global_dict.get("e", 0.0) + global_dict.get("sigma", 0.0) + global_dict.get("L", 0.0)
    q_draws = theil_global[:, 0] - stable
    q_med, q_lo, q_hi = ci(q_draws)
    df_shapley_global = pd.DataFrame([{
        "scope": "Global",
        "component": "Q",
        "baseline": global_dict.get("Q", np.nan),
        "median_independent": q_med,
        "lo95_independent": q_lo,
        "hi95_independent": q_hi,
    }])

    country_rows = []
    for gi in range(G):
        d = comp(m1_country, gi + 1)
        if not d:
            continue
        stable = d.get("e", 0.0) + d.get("sigma", 0.0) + d.get("L", 0.0)
        q_draws = theil_country[:, gi] - stable
        q_med, q_lo, q_hi = ci(q_draws)
        country_rows.append({
            "GLORIA_Index": gi + 1,
            "ISO3": gloria_iso3[gi],
            "Country": country_names[gi],
            "WBR": wbr[gi],
            "component": "Q",
            "baseline": d.get("Q", np.nan),
            "median_independent": q_med,
            "lo95_independent": q_lo,
            "hi95_independent": q_hi,
        })
    df_shapley_country = pd.DataFrame(country_rows)

    wbr_rows = []
    for ri, code in enumerate(WBR_CODES):
        d = comp(m1_wbr, code)
        if not d:
            continue
        stable = d.get("e", 0.0) + d.get("sigma", 0.0) + d.get("L", 0.0)
        q_draws = theil_wbr[:, ri, 0] - stable
        q_med, q_lo, q_hi = ci(q_draws)
        wbr_rows.append({
            "WBR": code,
            "Region": WBR_NAMES.get(code, code),
            "component": "Q",
            "baseline": d.get("Q", np.nan),
            "median_independent": q_med,
            "lo95_independent": q_lo,
            "hi95_independent": q_hi,
        })
    df_shapley_wbr = pd.DataFrame(wbr_rows)
    return {
        "ShapleyQ_Global": df_shapley_global,
        "ShapleyQ_Country": df_shapley_country,
        "ShapleyQ_WBR": df_shapley_wbr,
    }

shapley_tables = optional_shapley_q_tables()
print("Tables are ready.")

# Part 4. Export submission tables.
footprint_path = OUTPUT_DIR / "footprint_uncertainty_independent.xlsx"
with pd.ExcelWriter(footprint_path, engine="openpyxl") as writer:
    df_footprint_global.to_excel(writer, sheet_name="Global", index=False)
    df_footprint_wbr.to_excel(writer, sheet_name="WBR", index=False)
    df_footprint_country.to_excel(writer, sheet_name="Country", index=False)

inequality_path = OUTPUT_DIR / "inequality_uncertainty_independent.xlsx"
with pd.ExcelWriter(inequality_path, engine="openpyxl") as writer:
    df_inequality_global.to_excel(writer, sheet_name="Global", index=False)
    df_inequality_wbr.to_excel(writer, sheet_name="WBR", index=False)
    df_inequality_country.to_excel(writer, sheet_name="Country", index=False)
    df_sector_group.to_excel(writer, sheet_name="Sector_5class_FP", index=False)
    df_sector_class.to_excel(writer, sheet_name="Sector_10class_FP", index=False)
    for sheet, table in shapley_tables.items():
        table.to_excel(writer, sheet_name=sheet[:31], index=False)

scenario_path = OUTPUT_DIR / "scenario_uncertainty_independent.xlsx"
if len(df_scenario):
    with pd.ExcelWriter(scenario_path, engine="openpyxl") as writer:
        df_scenario.to_excel(writer, sheet_name="Scenarios", index=False)

combined_path = OUTPUT_DIR / "uncertainty_results_independent.xlsx"
with pd.ExcelWriter(combined_path, engine="openpyxl") as writer:
    df_footprint_global.to_excel(writer, sheet_name="Footprint_Global", index=False)
    df_footprint_wbr.to_excel(writer, sheet_name="Footprint_WBR", index=False)
    df_footprint_country.to_excel(writer, sheet_name="Footprint_Country", index=False)
    df_inequality_global.to_excel(writer, sheet_name="Inequality_Global", index=False)
    df_inequality_wbr.to_excel(writer, sheet_name="Inequality_WBR", index=False)
    df_inequality_country.to_excel(writer, sheet_name="Inequality_Country", index=False)
    df_sector_group.to_excel(writer, sheet_name="Sector_5class_FP", index=False)
    df_sector_class.to_excel(writer, sheet_name="Sector_10class_FP", index=False)
    if len(df_scenario):
        df_scenario.to_excel(writer, sheet_name="Scenarios", index=False)
    for sheet, table in shapley_tables.items():
        table.to_excel(writer, sheet_name=sheet[:31], index=False)

print("Saved:")
print(" ", footprint_path)
print(" ", inequality_path)
if len(df_scenario):
    print(" ", scenario_path)
print(" ", combined_path)
print(f"Finished in {(time.time() - t0) / 60:.1f} min")
